# 07 · PyTorch: fundamentos

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Com o scikit-learn, treinar um modelo é chamar `fit`. Com redes neurais, quem escreve o treino é você. Parece um retrocesso, e é o que dá a liberdade de montar qualquer arquitetura, qualquer função de perda e qualquer rotina de treino. Esta aula constrói essa habilidade do zero: do tensor ao laço de treino completo.

## O que você será capaz de fazer ao final

1. Criar e manipular tensores, cuidando de forma, tipo e dispositivo.
2. Explicar o que o *autograd* faz e calcular gradientes com `backward`.
3. Escrever a descida do gradiente à mão.
4. Definir um modelo com `nn.Module` e escolher função de perda e otimizador.
5. Escrever o laço de treino de cinco passos, com mini-lotes e validação.
6. Diagnosticar os erros mais comuns: forma, tipo, gradiente acumulado e *logits*.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import time
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEMENTE = 42
torch.manual_seed(SEMENTE)

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "· dispositivo:", dispositivo)

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})

> 📌 **GPU no Colab:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Esta aula roda bem em CPU; a GPU passa a importar na aula 08.

## 1. Por que PyTorch

O PyTorch é, no essencial, três coisas:

| Peça | O que é | Você já conhece como |
|---|---|---|
| **Tensor** | array de N dimensões que pode viver na GPU | `ndarray` do NumPy (aula 01) |
| **Autograd** | cálculo automático de derivadas de qualquer conta feita com tensores | novidade desta aula |
| **`torch.nn` e `torch.optim`** | camadas, funções de perda e otimizadores prontos | estimadores do scikit-learn, desmontados em peças |

A diferença de filosofia em relação ao scikit-learn:

| | scikit-learn | PyTorch |
|---|---|---|
| Modelo | escolhido de um catálogo | montado por você, camada a camada |
| Treino | `modelo.fit(X, y)` | um laço que você escreve |
| Onde brilha | tabelas | imagens, texto, áudio, e tudo que pede arquitetura sob medida |

## 2. Tensores

Um tensor tem três atributos que você vai consultar o tempo todo: **forma** (`shape`), **tipo** (`dtype`) e **dispositivo** (`device`).

In [ ]:
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(a)
print("forma:", a.shape, "· tipo:", a.dtype, "· dispositivo:", a.device)

In [ ]:
print(torch.zeros(2, 3))                     # criação, como no NumPy
print(torch.arange(5))
print(torch.linspace(0, 1, 5))

torch.manual_seed(0)                         # sorteios reprodutíveis
print(torch.randn(2, 2))                     # normal padrão

As operações seguem o NumPy, incluindo o *broadcasting* da aula 01. A diferença de vocabulário: o argumento `axis` chama-se `dim`.

In [ ]:
print(a + 10)                    # escalar com tensor
print(a * a)                     # elemento a elemento
print(a @ a.T)                   # produto de matrizes: (2,3) @ (3,2) -> (2,2)
print(a.mean(dim=0))             # média de cada coluna
print(a.sum().item())            # .item() converte um tensor de um elemento em número do Python

### Tipo: a primeira fonte de erros

O PyTorch usa `float32` como padrão para números reais (metade da memória do `float64`, e é o que as GPUs preferem). O NumPy usa `float64`. Misturar os dois gera erro.

In [ ]:
print(torch.tensor([1, 2, 3]).dtype)                 # inteiros  -> int64
print(torch.tensor([1.0, 2.0]).dtype)                # reais     -> float32
print(torch.from_numpy(np.array([1.0, 2.0])).dtype)  # vindo do NumPy -> float64 (!)

camada = nn.Linear(2, 1)                             # os pesos da camada são float32
entrada_numpy = torch.from_numpy(np.ones((1, 2)))    # float64
try:
    camada(entrada_numpy)
except RuntimeError as erro:
    print("\nRuntimeError:", erro)

print("Com a conversão:", camada(entrada_numpy.float()).shape)     # .float() converte para float32

> 📌 **Regra prática:** entradas e pesos em `float32`; rótulos de classes em `int64` (`.long()`); rótulos de problemas binários em `float32`.

### Ponte com o NumPy

`torch.from_numpy` e `.numpy()` **compartilham a memória** com o array original, como as visões da aula 01. Alterar um altera o outro.

In [ ]:
vetor = np.zeros(3)
tensor = torch.from_numpy(vetor)
vetor[0] = 99
print(tensor)                    # o tensor também mudou

print(a.numpy())                 # tensor -> NumPy (só funciona para tensores na CPU)

### Forma: a segunda fonte de erros

In [ ]:
v = torch.arange(6.0)
print(v.shape, "→", v.reshape(2, 3).shape)          # reorganiza
print(v.shape, "→", v.unsqueeze(1).shape)           # cria um eixo de tamanho 1 na posição 1: vetor -> coluna
print(v.unsqueeze(1).shape, "→", v.unsqueeze(1).squeeze(1).shape)     # remove o eixo de tamanho 1

`unsqueeze` e `squeeze` aparecem em todo código PyTorch, porque as camadas devolvem saídas com forma `(N, 1)` e os rótulos costumam vir com forma `(N,)`. Veremos na seção 9 o estrago que essa diferença causa.

### Dispositivo

Um tensor vive na CPU ou em uma GPU. **Todos os tensores de uma operação precisam estar no mesmo dispositivo.**

In [ ]:
a_no_dispositivo = a.to(dispositivo)          # copia para a GPU, se houver; na CPU, não faz nada
print(a_no_dispositivo.device)
print(a_no_dispositivo.cpu().numpy().shape)   # para voltar ao NumPy: primeiro .cpu()

### ✏️ Exercício 1 (★): padronizar colunas

O tensor `medidas` tem 100 linhas e 4 colunas em escalas muito diferentes. Crie `padronizado`, em que cada coluna tem média 0 e desvio padrão 1. Use `mean` e `std` com `dim=0`, sem laços.

In [ ]:
torch.manual_seed(0)
medidas = torch.rand(100, 4) * torch.tensor([1.0, 10.0, 100.0, 1000.0])
medidas[:3]

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
padronizado = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert padronizado.shape == (100, 4)
assert torch.allclose(padronizado.mean(dim=0), torch.zeros(4), atol=1e-5), "as médias das colunas devem ser zero"
assert torch.allclose(padronizado.std(dim=0), torch.ones(4), atol=1e-5), "os desvios padrão das colunas devem ser um"
print("Exercício 1 correto: (100, 4) menos (4,) dividido por (4,), com broadcasting.")

## 3. Autograd: derivadas automáticas

Treinar uma rede é ajustar milhões de números para reduzir um erro. Para saber em que direção mexer cada número, precisamos da **derivada do erro em relação a ele**. O *autograd* calcula essas derivadas sozinho.

Um exemplo que dá para conferir de cabeça: `y = x² + 3x`. A derivada é `2x + 3`; em `x = 2`, vale 7.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)     # requires_grad: "quero derivadas em relação a este tensor"
y = x ** 2 + 3 * x

print(y)                                      # o resultado lembra de que operação veio (grad_fn)
y.backward()                                  # percorre as contas de trás para a frente
print("dy/dx =", x.grad)                      # a derivada fica guardada em x.grad

**O que aconteceu.** Enquanto a conta era feita, o PyTorch registrou cada operação em um **grafo**: `x` foi elevado ao quadrado, `x` foi multiplicado por 3, os resultados foram somados. O `backward()` percorre esse grafo do resultado até as entradas, aplicando a regra da cadeia. É a *retropropagação*.

O grafo é montado **durante a execução**, a cada vez. Por isso o código do modelo pode ter `if`, `for` e qualquer outra construção do Python.

### 🔮 Preveja: gradientes se acumulam

Vamos refazer a mesma conta e chamar `backward()` de novo, sem mexer em `x.grad`. Que valor aparece?

In [ ]:
y = x ** 2 + 3 * x
y.backward()
print("depois do segundo backward:", x.grad)

x.grad.zero_()                                # zera no lugar (o sublinhado final indica operação no lugar)
print("depois de zerar:           ", x.grad)

O segundo `backward` **somou** 7 ao valor que já estava lá. O PyTorch acumula gradientes de propósito (é útil em algumas técnicas), e deixa para você a tarefa de zerá-los a cada passo. Esquecer disso é o erro mais clássico de quem começa.

### Desligando o registro

Quando não vamos calcular derivadas (ao avaliar o modelo, ao atualizar os pesos), desligamos o registro com `torch.no_grad()`. Fica mais rápido e gasta menos memória.

In [ ]:
with torch.no_grad():
    z = x ** 2
print("z guarda histórico?", z.requires_grad)

w = torch.tensor([1.0, -2.0, 0.5], requires_grad=True)      # com vários parâmetros, o gradiente tem a mesma forma deles
perda = (w ** 2).sum()
perda.backward()
print("gradiente de soma(w²):", w.grad, "(esperado: 2w)")

> 📌 `backward()` só pode ser chamado em um **escalar** (um número): a perda. O gradiente de cada parâmetro tem sempre a mesma forma do parâmetro.

### ✏️ Exercício 2 (★★): autograd contra a fórmula

Para a perda `média((X @ w − y)²)`, o gradiente em relação a `w` tem fórmula conhecida: `(2 / n) · Xᵀ (X @ w − y)`.

- Calcule a perda, chame `backward` e guarde uma cópia do gradiente em `grad_auto` (use `.clone()`).
- Calcule a fórmula diretamente e guarde em `grad_formula`.

In [ ]:
# ✏️ Exercício 2
torch.manual_seed(1)
X_ex2, y_ex2 = torch.randn(50, 3), torch.randn(50)
w_ex2 = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)

# SEU CÓDIGO AQUI
grad_auto = ...
grad_formula = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert grad_auto.shape == (3,) and grad_formula.shape == (3,), "o gradiente tem a mesma forma de w"
assert grad_auto.abs().sum() > 0.1, "o gradiente está zerado: o backward foi chamado?"
assert torch.allclose(grad_auto, grad_formula.detach(), atol=1e-5), "autograd e fórmula não coincidem"
print("Exercício 2 correto:", grad_auto)

## 4. Descida do gradiente à mão

Com o gradiente em mãos, o aprendizado é uma regra de uma linha:

> **parâmetro ← parâmetro − taxa de aprendizado × gradiente**

O gradiente aponta a direção em que a perda mais cresce; andamos um passo pequeno na direção oposta, e repetimos.

Vamos ajustar uma reta a pontos gerados por `y = 2x − 1` com ruído. Os parâmetros começam em zero.

In [ ]:
torch.manual_seed(SEMENTE)
x_reta = torch.linspace(-2, 2, 100).unsqueeze(1)             # forma (100, 1)
y_reta = 2 * x_reta - 1 + 0.3 * torch.randn(100, 1)          # forma (100, 1)

w = torch.zeros(1, requires_grad=True)                       # inclinação
b = torch.zeros(1, requires_grad=True)                       # intercepto
taxa = 0.1
historico = []

for passo in range(100):
    previsao = x_reta * w + b                                # 1. previsão com os parâmetros atuais
    perda = ((previsao - y_reta) ** 2).mean()                # 2. perda: erro quadrático médio
    perda.backward()                                         # 3. gradientes de w e de b
    with torch.no_grad():                                    # 4. atualização (não faz parte do grafo)
        w -= taxa * w.grad
        b -= taxa * b.grad
    w.grad.zero_()                                           # 5. zera para o próximo passo
    b.grad.zero_()
    historico.append(perda.item())

print(f"w = {w.item():.3f} (verdadeiro: 2) · b = {b.item():.3f} (verdadeiro: -1) · perda final = {historico[-1]:.4f}")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(10.5, 3.3))
eixos[0].plot(historico, color=AZUL)
eixos[0].set_xlabel("passo"); eixos[0].set_ylabel("perda"); eixos[0].set_title("A perda cai depressa e estabiliza no nível do ruído")
eixos[1].scatter(x_reta, y_reta, s=10, color=CINZA, alpha=0.6)
eixos[1].plot(x_reta, (x_reta * w + b).detach(), color=LARANJA, lw=2)
eixos[1].set_xlabel("x"); eixos[1].set_title("A reta ajustada")
plt.tight_layout()

A perda estabiliza perto de 0,09, que é a variância do ruído (0,3²): o que resta de erro não depende de `x`.

`.detach()` aparece no gráfico porque o Matplotlib não aceita tensores que guardam histórico; ele devolve o mesmo tensor, desligado do grafo.

### A taxa de aprendizado

🔮 **Preveja:** o que acontece com uma taxa muito pequena? E com uma muito grande?

In [ ]:
def perda_final(taxa, passos=100):
    w = torch.zeros(1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    for _ in range(passos):
        perda = ((x_reta * w + b - y_reta) ** 2).mean()
        perda.backward()
        with torch.no_grad():
            w -= taxa * w.grad
            b -= taxa * b.grad
        w.grad.zero_(); b.grad.zero_()
    return perda.item()


pd.DataFrame({"taxa": [0.001, 0.01, 0.1, 0.5, 0.8]}).assign(perda_apos_100_passos=lambda t: t["taxa"].map(perda_final))

- **Taxa pequena demais** (0,001): cada passo é minúsculo e, depois de 100 passos, a perda ainda está em 3,9. Com 0,01 falta pouco, e ainda não chegou.
- **Taxa adequada** (0,1 e 0,5): chega ao nível do ruído (0,087).
- **Taxa grande demais** (0,8): cada passo ultrapassa o ponto ótimo por uma distância maior que a anterior, e a perda **explode**.

A taxa de aprendizado é o hiperparâmetro mais importante do treino de redes. Quando a perda vira `nan` ou cresce sem parar, a primeira providência é reduzi-la.

### ✏️ Exercício 3 (★★): uma parábola, à mão

Os dados abaixo seguem `y = a·x² + b·x + c` com ruído. Ajuste os três parâmetros por descida do gradiente, **sem** `nn` nem `optim`, começando em zero, com taxa 0,01 e 3000 passos. Guarde os tensores em `a_ex3`, `b_ex3` e `c_ex3`.

In [ ]:
torch.manual_seed(3)
x_par = torch.linspace(-3, 3, 200).unsqueeze(1)
y_par = 0.5 * x_par ** 2 - x_par + 2 + 0.2 * torch.randn(200, 1)

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
a_ex3 = ...
b_ex3 = ...
c_ex3 = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_perda = ((a_ex3 * x_par ** 2 + b_ex3 * x_par + c_ex3 - y_par) ** 2).mean().item()
assert abs(a_ex3.item() - 0.5) < 0.05 and abs(b_ex3.item() + 1) < 0.05 and abs(c_ex3.item() - 2) < 0.15, "os parâmetros estão longe de a=0,5, b=-1, c=2"
assert _perda < 0.06, "a perda ainda está alta: mais passos? gradientes zerados a cada passo?"
print(f"Exercício 3 correto: a = {a_ex3.item():.3f} · b = {b_ex3.item():.3f} · c = {c_ex3.item():.3f} · perda = {_perda:.4f}")

## 5. As mesmas cinco linhas, com as ferramentas da biblioteca

Escrever `w -= taxa * w.grad` para cada parâmetro não escala para milhões deles. O PyTorch oferece três peças que organizam o que fizemos à mão:

| À mão | Com a biblioteca |
|---|---|
| `w` e `b` criados com `requires_grad=True` | **`nn.Linear`**: uma camada que já contém pesos e viés |
| `((previsao - y) ** 2).mean()` | **`nn.MSELoss()`**: função de perda pronta |
| `w -= taxa * w.grad` e `w.grad.zero_()` | **otimizador**: `otimizador.step()` e `otimizador.zero_grad()` |

In [ ]:
torch.manual_seed(SEMENTE)
modelo = nn.Linear(in_features=1, out_features=1)             # y = w·x + b, com w e b sorteados
funcao_perda = nn.MSELoss()
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.1)     # SGD: a regra de atualização da seção 4

for passo in range(100):
    otimizador.zero_grad()                     # 1. zera os gradientes do passo anterior
    previsao = modelo(x_reta)                  # 2. previsão (forward)
    perda = funcao_perda(previsao, y_reta)     # 3. perda
    perda.backward()                           # 4. gradientes (backward)
    otimizador.step()                          # 5. atualiza os parâmetros

print(f"w = {modelo.weight.item():.3f} · b = {modelo.bias.item():.3f} · perda final = {perda.item():.4f}")

O resultado é o mesmo da seção 4. Estas cinco linhas são o **laço de treino**; elas aparecem, quase idênticas, em qualquer projeto com PyTorch, de uma reta a um modelo de linguagem.

> 📌 **Os cinco passos:** zerar · prever · medir a perda · retropropagar · atualizar.

### Definindo o seu próprio modelo: `nn.Module`

Todo modelo em PyTorch é uma classe que herda de `nn.Module` (as classes foram revistas na aula 00). Ela tem duas partes:

- `__init__`: cria as camadas, que guardam os parâmetros;
- `forward`: descreve a conta que leva a entrada à saída.

In [ ]:
class RegressaoLinear(nn.Module):
    def __init__(self, n_entradas):
        super().__init__()                           # obrigatório: inicializa a maquinaria do nn.Module
        self.linear = nn.Linear(n_entradas, 1)       # camadas viram atributos

    def forward(self, x):
        return self.linear(x)                        # chamada quando escrevemos modelo(x)


modelo = RegressaoLinear(n_entradas=3)
print(modelo)
for nome, parametro in modelo.named_parameters():    # tudo que foi criado no __init__ é encontrado automaticamente
    print(f"{nome:<15}{str(tuple(parametro.shape)):<10}requires_grad={parametro.requires_grad}")
print("total de parâmetros:", sum(p.numel() for p in modelo.parameters()))

Chamamos `modelo(x)`, e nunca `modelo.forward(x)` diretamente: a chamada pelo objeto executa etapas internas adicionais.

Para modelos que são uma simples fila de camadas, existe o atalho `nn.Sequential`, que usaremos na seção 7.

## 6. Classificação: *logits* e funções de perda

Em classificação, a última camada de uma rede **não** produz probabilidades. Ela produz números sem limite, positivos ou negativos, chamados ***logits*** (em problemas binários, são as log-chances da aula 06). A conversão em probabilidade fica por conta da função de perda, que a faz de forma numericamente estável.

| Problema | Saída do modelo | Função de perda | Rótulos | Para obter probabilidades |
|---|---|---|---|---|
| Regressão | 1 número | `nn.MSELoss()` | `float32`, mesma forma da saída | — |
| Binária | 1 *logit* | `nn.BCEWithLogitsLoss()` | `float32` (0.0 ou 1.0), mesma forma da saída | `torch.sigmoid(logit)` |
| Multiclasse (K classes) | K *logits* | `nn.CrossEntropyLoss()` | `int64`, forma `(N,)`, com o índice da classe | `torch.softmax(logits, dim=1)` |

In [ ]:
logits = torch.tensor([-2.0, 0.0, 3.0])
alvos = torch.tensor([0.0, 1.0, 1.0])

print("probabilidades:", torch.sigmoid(logits))
print("perda pronta:  ", nn.BCEWithLogitsLoss()(logits, alvos).item())

p = torch.sigmoid(logits)                                                    # a mesma conta, por extenso: a perda logarítmica
print("perda à mão:   ", (-(alvos * p.log() + (1 - alvos) * (1 - p).log())).mean().item())

> ⚠️ **Não aplique `sigmoid` ou `softmax` dentro do modelo** quando usar essas funções de perda. Elas já fazem a conversão; aplicar duas vezes não gera erro, apenas um modelo que aprende mal.

## 7. Camadas ocultas: o problema das duas luas

Uma camada linear só traça fronteiras retas. O conjunto abaixo tem duas classes em forma de meia-lua entrelaçadas; nenhuma reta as separa.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

X_luas_np, y_luas_np = make_moons(n_samples=1000, noise=0.2, random_state=SEMENTE)
X_luas_teste_np, y_luas_teste_np = make_moons(n_samples=1000, noise=0.2, random_state=SEMENTE + 1)

X_luas = torch.tensor(X_luas_np, dtype=torch.float32)                 # (1000, 2)
y_luas = torch.tensor(y_luas_np, dtype=torch.float32)                 # (1000,)
X_luas_teste = torch.tensor(X_luas_teste_np, dtype=torch.float32)


def desenhar_fronteira(prever, ax, titulo):
    """`prever` recebe um array (N, 2) e devolve a probabilidade da classe 1 para cada ponto."""
    g1, g2 = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
    prob = prever(np.column_stack([g1.ravel(), g2.ravel()])).reshape(g1.shape)
    ax.contourf(g1, g2, prob, levels=[0, 0.5, 1], colors=[AZUL, LARANJA], alpha=0.15)
    ax.contour(g1, g2, prob, levels=[0.5], colors=CINZA, linewidths=1)
    ax.scatter(*X_luas_teste_np.T, c=np.where(y_luas_teste_np == 1, LARANJA, AZUL), s=6, alpha=0.7)
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([])


linear = LogisticRegression().fit(X_luas_np, y_luas_np)
fig, ax = plt.subplots(figsize=(5.2, 3.4))
desenhar_fronteira(lambda pontos: linear.predict_proba(pontos)[:, 1], ax, f"Modelo linear: acurácia {linear.score(X_luas_teste_np, y_luas_teste_np):.1%} no teste")

A saída é empilhar camadas, com uma **função de ativação** não linear entre elas. Sem a ativação, duas camadas lineares em sequência equivalem a uma só (a composição de retas é uma reta). A ativação mais usada é a **ReLU**: `max(0, x)`.

Uma rede com camadas lineares e ativações chama-se **MLP** (*multilayer perceptron*). Com `nn.Sequential`, escreve-se assim:

In [ ]:
torch.manual_seed(SEMENTE)
rede = nn.Sequential(
    nn.Linear(2, 16),        # 2 entradas -> 16 neurônios na camada oculta
    nn.ReLU(),               # não linearidade
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),        # 1 logit de saída
)
print(rede)
print("parâmetros:", sum(p.numel() for p in rede.parameters()))
print("saída para 5 pontos:", rede(X_luas[:5]).shape)        # (5, 1): um logit por ponto

### ✏️ Exercício 4 (★★★): seu primeiro laço de treino

Treine uma rede para separar as duas luas. Você escreve o laço inteiro.

- Crie `modelo_luas` (pode usar a arquitetura acima) e um otimizador `torch.optim.Adam` com `lr=0.01`.
- Use `nn.BCEWithLogitsLoss()`. Atenção às formas: a rede devolve `(1000, 1)` e `y_luas` tem forma `(1000,)`.
- Faça 500 passos usando todos os pontos de `X_luas` em cada passo.

In [ ]:
# ✏️ Exercício 4
torch.manual_seed(SEMENTE)
# SEU CÓDIGO AQUI
modelo_luas = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
with torch.no_grad():
    _logits = modelo_luas(X_luas_teste).reshape(-1)
_acuracia = ((_logits > 0).numpy() == y_luas_teste_np).mean()
assert _acuracia > 0.95, f"acurácia de {_acuracia:.1%} no teste: confira os cinco passos do laço e as formas"

def _prever(pontos):
    with torch.no_grad():
        return torch.sigmoid(modelo_luas(torch.tensor(pontos, dtype=torch.float32))).reshape(-1).numpy()

fig, ax = plt.subplots(figsize=(5.2, 3.4))
desenhar_fronteira(_prever, ax, f"Sua rede: acurácia {_acuracia:.1%} no teste")
print("Exercício 4 correto.")

## 8. O modelo da aula 05, em PyTorch

Voltamos às avaliações da B2W e ao alvo de sempre: o cliente **não recomenda** o produto. Para caber em tensores densos, representamos cada texto pela presença das 1000 palavras mais frequentes.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend"])
textos = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
alvo = (df["recommend_to_a_friend"] == "No").astype(int)

# As mesmas três partes da aula 06: ajuste, validação e teste (o teste não é usado hoje)
t_treino, t_teste, y_treino, y_teste = train_test_split(textos, alvo, test_size=0.2, stratify=alvo, random_state=SEMENTE)
t_ajuste, t_valid, y_ajuste, y_valid = train_test_split(t_treino, y_treino, test_size=0.25, stratify=y_treino, random_state=SEMENTE)

vetorizador = CountVectorizer(max_features=1000, binary=True, dtype=np.float32)
A = vetorizador.fit_transform(t_ajuste)          # vocabulário aprendido só no ajuste
V = vetorizador.transform(t_valid)

referencia = LogisticRegression(max_iter=1000).fit(A, y_ajuste)
auc_referencia = roc_auc_score(y_valid, referencia.predict_proba(V)[:, 1])
print(f"{A.shape[0]} avaliações de ajuste × {A.shape[1]} palavras · regressão logística do scikit-learn: AUC {auc_referencia:.4f} na validação")

In [ ]:
# De matrizes esparsas para tensores densos, no dispositivo de treino
X_aj = torch.from_numpy(A.toarray()).to(dispositivo)
X_va = torch.from_numpy(V.toarray()).to(dispositivo)
y_aj = torch.tensor(y_ajuste.to_numpy(), dtype=torch.float32, device=dispositivo)
y_va = torch.tensor(y_valid.to_numpy(), dtype=torch.float32, device=dispositivo)

print(X_aj.shape, X_aj.dtype, f"· {X_aj.nelement() * X_aj.element_size() / 1e6:.0f} MB · alvo:", y_aj.shape, y_aj.dtype)

### Mini-lotes

Na seção 4, cada passo usou todos os pontos. Com 79 mil avaliações (ou 79 milhões), isso é lento e pode não caber na memória da GPU. A prática universal é o treino em **mini-lotes**:

| Termo | Significado |
|---|---|
| **Lote** (*batch*) | um pedaço dos dados, por exemplo 256 exemplos |
| **Passo** (*iteração*) | uma atualização dos parâmetros, feita com um lote |
| **Época** | uma passagem completa pelos dados de ajuste |

A cada época os dados são **embaralhados**, para que os lotes mudem. O gradiente de um lote é uma estimativa ruidosa do gradiente completo; o ruído atrapalha pouco e os passos ficam centenas de vezes mais baratos. É a descida do gradiente **estocástica** (SGD).

### Treino e validação em uma função

A função abaixo reúne tudo: épocas, embaralhamento, mini-lotes, os cinco passos e a medição na validação ao fim de cada época. Leia com calma; ela é o esqueleto de todo treino que você fará.

In [ ]:
funcao_perda = nn.BCEWithLogitsLoss()


def avaliar(modelo, X_, y_):
    """Perda e AUC de um modelo em um conjunto, sem calcular gradientes."""
    modelo.eval()                                         # modo de avaliação (importa para camadas como o dropout)
    with torch.no_grad():
        logits = modelo(X_).squeeze(1)
    return funcao_perda(logits, y_).item(), roc_auc_score(y_.cpu().numpy(), logits.cpu().numpy())


def treinar(modelo, epocas, taxa=1e-3, lote=256):
    otimizador = torch.optim.Adam(modelo.parameters(), lr=taxa)       # Adam: SGD com passo adaptado a cada parâmetro
    historico = []
    for epoca in range(1, epocas + 1):
        modelo.train()                                    # modo de treino
        ordem = torch.randperm(len(X_aj), device=dispositivo)         # embaralha os índices
        soma = 0.0
        for inicio in range(0, len(X_aj), lote):
            indices = ordem[inicio:inicio + lote]
            xb, yb = X_aj[indices], y_aj[indices]         # um mini-lote

            otimizador.zero_grad()                        # 1. zerar
            logits = modelo(xb).squeeze(1)                # 2. prever      (lote, 1) -> (lote,)
            perda = funcao_perda(logits, yb)              # 3. medir a perda
            perda.backward()                              # 4. retropropagar
            otimizador.step()                             # 5. atualizar

            soma += perda.item() * len(indices)
        perda_valid, auc_valid = avaliar(modelo, X_va, y_va)
        historico.append({"epoca": epoca, "perda_ajuste": soma / len(X_aj), "perda_valid": perda_valid, "auc_valid": auc_valid})
        print(f"época {epoca:>2} · perda de ajuste {soma / len(X_aj):.4f} · perda de validação {perda_valid:.4f} · AUC de validação {auc_valid:.4f}")
    return pd.DataFrame(historico)

Uma regressão logística é uma rede sem camada oculta: uma camada linear que leva as 1000 entradas a 1 *logit*.

In [ ]:
torch.manual_seed(SEMENTE)
logistica = nn.Linear(1000, 1).to(dispositivo)            # o modelo também precisa estar no dispositivo

inicio = time.perf_counter()
historico_logistica = treinar(logistica, epocas=8)
print(f"\n{time.perf_counter() - inicio:.1f} s · scikit-learn nas mesmas colunas: AUC {auc_referencia:.4f}")

Em oito épocas, a regressão logística escrita em PyTorch se aproxima da do scikit-learn, e ainda melhora um pouco a cada época: é o mesmo modelo, encontrado por outro caminho. O scikit-learn usa um otimizador que vai direto ao ponto ótimo deste problema específico; o PyTorch usa passos pequenos, que servem para qualquer modelo.

In [ ]:
pesos = pd.Series(logistica.weight.detach().cpu().numpy()[0], index=vetorizador.get_feature_names_out())
print("Empurram para 'não recomenda':", ", ".join(pesos.nlargest(8).index))
print("Empurram para 'recomenda':    ", ", ".join(pesos.nsmallest(8).index))

### ✏️ Exercício 5 (★★): uma camada oculta

Defina a classe `MLP`, herdando de `nn.Module`, com: uma camada linear de 1000 entradas para 64 neurônios, ReLU, e uma camada linear de 64 para 1. Depois crie `modelo_ex5` (no dispositivo) e treine por 10 épocas com a função `treinar`, guardando o histórico em `historico_ex5`.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
class MLP(nn.Module):
    ...


modelo_ex5 = ...
historico_ex5 = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert isinstance(modelo_ex5, nn.Module) and type(modelo_ex5).__name__ == "MLP"
assert sum(p.numel() for p in modelo_ex5.parameters()) == 1000 * 64 + 64 + 64 + 1, "o número de parâmetros não corresponde a 1000 -> 64 -> 1"
assert modelo_ex5(X_va[:5]).shape == (5, 1), "a saída deve ter um logit por exemplo"
assert len(historico_ex5) == 10 and historico_ex5["auc_valid"].iloc[-1] > 0.955
print(f"Exercício 5 correto: AUC de validação {historico_ex5['auc_valid'].iloc[-1]:.4f} (regressão logística: {auc_referencia:.4f}).")

### O que as curvas contam

Treinamos de novo uma rede com a mesma arquitetura, agora por 15 épocas, para observar as curvas.

In [ ]:
torch.manual_seed(SEMENTE)
mlp = nn.Sequential(nn.Linear(1000, 64), nn.ReLU(), nn.Linear(64, 1)).to(dispositivo)
historico_mlp = treinar(mlp, epocas=15)

In [ ]:
melhor_epoca = int(historico_mlp.loc[historico_mlp["perda_valid"].idxmin(), "epoca"])

fig, eixos = plt.subplots(1, 2, figsize=(11, 3.4))
eixos[0].plot(historico_mlp["epoca"], historico_mlp["perda_ajuste"], color=CINZA, label="ajuste")
eixos[0].plot(historico_mlp["epoca"], historico_mlp["perda_valid"], color=LARANJA, label="validação")
eixos[0].axvline(melhor_epoca, color=LARANJA, ls=":", lw=1)
eixos[0].set_xlabel("época"); eixos[0].set_ylabel("perda"); eixos[0].legend(frameon=False)
eixos[0].set_title(f"A perda de validação é mínima na época {melhor_epoca}")
eixos[1].plot(historico_mlp["epoca"], historico_mlp["auc_valid"], color=AZUL, label="MLP")
eixos[1].plot(historico_logistica["epoca"], historico_logistica["auc_valid"], color=CINZA, label="regressão logística (PyTorch)")
eixos[1].axhline(auc_referencia, color=CINZA, ls="--", lw=0.8)
eixos[1].set_xlabel("época"); eixos[1].set_ylabel("AUC de validação"); eixos[1].legend(frameon=False, fontsize=8)
eixos[1].set_title("A camada oculta rende alguns milésimos de AUC")
plt.tight_layout()
print(f"MLP: melhor perda de validação na época {melhor_epoca}; AUC máxima {historico_mlp['auc_valid'].max():.4f} · tracejado: scikit-learn ({auc_referencia:.4f})")

Duas leituras:

- A camada oculta melhora um pouco o resultado (a rede pode combinar palavras), mas o ganho é de milésimos. Neste problema, a aula 06 já tinha mostrado que um modelo linear vai longe.
- A perda de **ajuste** cai a cada época; a de **validação** para de cair por volta da época 6 e passa a subir devagar. É a mesma assinatura de sobreajuste que vimos no boosting, e pede o mesmo remédio: acompanhar a validação e ficar com o modelo da melhor época.

(Na época 1 a perda de ajuste aparece acima da de validação porque é a média ao longo da época, enquanto o modelo ainda estava aprendendo; a de validação é medida no fim.)

Guardar o melhor modelo, parar na hora certa, carregar dados em lotes sem escrever índices à mão e usar GPU de verdade são os temas da aula 08.

## 9. Caça aos erros

Os erros de PyTorch que mais custam tempo são os que **não geram mensagem de erro**. O mais traiçoeiro envolve formas.

In [ ]:
previsoes = torch.tensor([[1.0], [2.0], [3.0]])        # forma (3, 1), como sai de uma camada
alvos_reg = torch.tensor([1.0, 2.0, 3.0])              # forma (3,), como costuma vir dos dados

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    errada = nn.MSELoss()(previsoes, alvos_reg)
certa = nn.MSELoss()(previsoes.squeeze(1), alvos_reg)

print("previsões idênticas aos alvos; a perda deveria ser zero.")
print("perda com formas (3, 1) e (3,):", errada.item())
print("perda com formas (3,) e (3,):  ", certa.item())
print("\nO PyTorch avisa, mas não interrompe:", str(avisos[0].message)[:120], "...")

Com formas `(3, 1)` e `(3,)`, o *broadcasting* monta uma tabela 3 × 3 com todas as diferenças entre previsões e alvos, e a perda é calculada sobre nove números que não fazem sentido. O treino roda, a perda cai um pouco e o modelo converge para prever a média. **Confira as formas antes da perda**, sempre.

### ✏️ Exercício 6 (★★): três erros em dez linhas

O código abaixo deveria ajustar a reta da seção 4, e tem três erros: um interrompe a execução e dois são silenciosos. Corrija-os. O laço deve continuar com 200 passos e taxa 0,1.

In [ ]:
# ✏️ Exercício 6
torch.manual_seed(SEMENTE)
# SEU CÓDIGO AQUI
x_ex6 = torch.from_numpy(np.linspace(-2, 2, 100)).unsqueeze(1)      # os mesmos valores de x_reta, vindos do NumPy
modelo_ex6 = nn.Linear(1, 1)
otimizador_ex6 = torch.optim.SGD(modelo_ex6.parameters(), lr=0.1)
alvo_ex6 = y_reta.squeeze(1)

for passo in range(200):
    previsao_ex6 = modelo_ex6(x_ex6)
    perda_ex6 = nn.MSELoss()(previsao_ex6, alvo_ex6)
    perda_ex6.backward()
    otimizador_ex6.step()

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
assert abs(modelo_ex6.weight.item() - 2) < 0.1, f"inclinação {modelo_ex6.weight.item():.2f}; o esperado é perto de 2"
assert abs(modelo_ex6.bias.item() + 1) < 0.1, f"intercepto {modelo_ex6.bias.item():.2f}; o esperado é perto de -1"
assert perda_ex6.item() < 0.12, "a perda final deveria ficar perto de 0,09"
print(f"Exercício 6 correto: w = {modelo_ex6.weight.item():.3f} · b = {modelo_ex6.bias.item():.3f} · perda = {perda_ex6.item():.4f}")

## 10. Para levar

### Guardar e carregar um modelo

O que se salva são os parâmetros (`state_dict`), e não o objeto. Para carregar, recria-se o modelo com a mesma arquitetura.

In [ ]:
torch.save(mlp.state_dict(), "mlp_avaliacoes.pt")

copia = nn.Sequential(nn.Linear(1000, 64), nn.ReLU(), nn.Linear(64, 1)).to(dispositivo)      # mesma arquitetura
copia.load_state_dict(torch.load("mlp_avaliacoes.pt", map_location=dispositivo, weights_only=True))

print(f"arquivo: {Path('mlp_avaliacoes.pt').stat().st_size / 1e3:.0f} kB · AUC da cópia: {avaliar(copia, X_va, y_va)[1]:.4f} · AUC do original: {avaliar(mlp, X_va, y_va)[1]:.4f}")

### Erros comuns

| Sintoma | Causa provável | Correção |
|---|---|---|
| `mat1 and mat2 must have the same dtype` | dados em `float64` vindos do NumPy ou do pandas | `.float()` ou `dtype=torch.float32` |
| `Expected all tensors to be on the same device` | modelo na GPU e dados na CPU (ou o contrário) | `.to(dispositivo)` em ambos |
| `mat1 and mat2 shapes cannot be multiplied` | número de entradas da camada diferente do número de colunas | conferir `X.shape` e `in_features` |
| Perda cai pouco e o modelo prevê a média | formas `(N, 1)` e `(N,)` na perda | `squeeze(1)` na saída, ou `unsqueeze(1)` no alvo |
| Perda oscila ou explode desde o início | faltou `zero_grad()`, ou a taxa está alta demais | zerar a cada passo; reduzir a taxa |
| Perda vira `nan` | taxa alta demais; divisão por zero ou `log(0)` em conta própria | reduzir a taxa; usar as perdas com *logits* |
| Modelo aprende mal em classificação | `sigmoid` ou `softmax` dentro do modelo, antes da perda | devolver *logits* |
| `CrossEntropyLoss` reclama do alvo | rótulos em `float` ou em *one-hot* | índices das classes em `int64` |
| Memória cresce a cada época | guardar `perda` (com grafo) em uma lista | guardar `perda.item()` |
| Resultado da validação muda a cada chamada | faltou `modelo.eval()` | `eval()` para avaliar, `train()` para treinar |

### Resumo

- **Tensor** = array do NumPy com tipo, forma e dispositivo. Padrão `float32`; rótulos de classes em `int64`.
- **Autograd** registra as contas e calcula as derivadas com `backward()`. Os gradientes se acumulam: é preciso zerá-los.
- **Descida do gradiente**: parâmetro ← parâmetro − taxa × gradiente. A taxa de aprendizado é o primeiro hiperparâmetro a ajustar.
- **`nn.Module`** guarda as camadas no `__init__` e a conta no `forward`. **`nn.Sequential`** é o atalho para camadas em fila.
- **O laço de treino** tem cinco passos: zerar, prever, medir a perda, retropropagar, atualizar.
- O modelo devolve ***logits***; a função de perda cuida da conversão.
- Treina-se em **mini-lotes**, por várias **épocas**, medindo a **validação** ao fim de cada uma.

### O que vem a seguir

O notebook 08 transforma este esqueleto em um treino completo: `Dataset` e `DataLoader`, GPU, redes convolucionais para imagens, parada antecipada, *checkpoints* e registro de experimentos.

### Para casa

A lista de exercícios está em `07_casa.ipynb`.